# Análise Global de Mudanças Climáticas e Eventos Extremos
### Sistemas Distribuídos — UEL 2026
---
Este notebook realiza o processamento e análise de dados históricos de temperatura global
utilizando **Apache Spark** em cluster distribuído via Docker.

**Dataset principal:** GlobalLandTemperaturesByCity.csv (Berkeley Earth / Kaggle)  
**Dataset secundário:** owid-co2-data.csv (Our World in Data)

**Perguntas respondidas:**
1. Média Móvel de Temperatura por Década
2. Anomalias Locais — 10 Anos Mais Quentes por Continente
3. Cidades em Risco — Maior Desvio Padrão de Temperatura
4. Correlação de Estações em Zonas Tropicais
5. Qualidade de Dados — Filtro de Incerteza
6. Correlação entre Emissões de CO₂ e Aquecimento (Join)
7. Ranking de Aceleração Térmica (Window Functions)
8. Previsão de Tendência com Regressão Linear (MLlib)


## 1. Imports

In [1]:
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, year, month, to_date, avg, stddev,
    max as _max, min as _min, regexp_replace,
    expr, count, when, lag, round as _round,
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

# Marca o início do pipeline inteiro
_tempo_inicio_geral = time.time()
print("Imports carregados com sucesso.")

Imports carregados com sucesso.


## 2. Inicialização da Sessão Spark

Criamos uma `SparkSession` apontando para o master do cluster Docker.
O `setLogLevel("WARN")` reduz o volume de logs, exibindo apenas avisos e erros.


In [2]:
spark = (
    SparkSession.builder
    .appName("AnaliseClimaticaGlobal")
    .master("spark://spark-master:7077")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "/opt/spark/spark-events")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Sessão Spark iniciada com sucesso!")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 00:14:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sessão Spark iniciada com sucesso!


## 3. ETL — Ingestão e Limpeza de Dados

### 3.1 Leitura da base de temperaturas

Utilizamos o arquivo `GlobalLandTemperaturesByCity.csv` da Berkeley Earth.
O Spark infere o schema automaticamente com `inferSchema=True`.


In [3]:
_t = time.time()

df = spark.read.csv(
    "/opt/spark-data/input/GlobalLandTemperaturesByCity.csv",
    header=True,
    inferSchema=True,
)

print(f"Total de registros brutos: {df.count():,}")
df.printSchema()
df.show(5)
print(f"[Tempo] Leitura CSV temperatura: {time.time() - _t:.2f}s")

Total de registros brutos: 8,599,212
root
 |-- dt: date (nullable = true)
 |-- AverageTemperature: double (nullable = true)
 |-- AverageTemperatureUncertainty: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)

+----------+------------------+-----------------------------+-----+-------+--------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|
+----------+------------------+-----------------------------+-----+-------+--------+---------+
|1743-11-01|             6.068|           1.7369999999999999|Århus|Denmark|  57.05N|   10.33E|
|1743-12-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   10.33E|
|1744-01-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   10.33E|
|1744-02-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   1

### 3.2 Limpeza e transformações

Realizamos as seguintes operações:
- Conversão da coluna `dt` de String para `DateType`
- Extração das colunas `Year` e `Month` para facilitar as agregações
- Remoção de linhas com `AverageTemperature` nula
- Limpeza das coordenadas — remoção de letras N/S/E/W e conversão para `double`


In [4]:
_t = time.time()

df_limpo = (
    df.withColumn("dt", to_date(col("dt")))
    .withColumn("Year", year(col("dt")))
    .withColumn("Month", month(col("dt")))
    .dropna(subset=["AverageTemperature"])
    .withColumn("Latitude",  regexp_replace(col("Latitude"),  "[NSEW]", "").cast("double"))
    .withColumn("Longitude", regexp_replace(col("Longitude"), "[NSEW]", "").cast("double"))
)

print(f"Registros após remoção de nulos: {df_limpo.count():,}")
df_limpo.show(5)
print(f"[Tempo] Limpeza e transformações: {time.time() - _t:.2f}s")

Registros após remoção de nulos: 8,235,082
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|Year|Month|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
|1743-11-01|             6.068|           1.7369999999999999|Århus|Denmark|   57.05|    10.33|1743|   11|
|1744-04-01|5.7879999999999985|           3.6239999999999997|Århus|Denmark|   57.05|    10.33|1744|    4|
|1744-05-01|            10.644|           1.2830000000000001|Århus|Denmark|   57.05|    10.33|1744|    5|
|1744-06-01|14.050999999999998|                        1.347|Århus|Denmark|   57.05|    10.33|1744|    6|
|1744-07-01|            16.082|                        1.396|Århus|Denmark|   57.05|    10.33|1744|    7|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
onl

### 3.3 Cache do DataFrame limpo

O `.cache()` instrui o Spark a **guardar o DataFrame em memória** após a primeira vez
que ele for computado. Sem cache, cada nova ação (count, show, groupBy, etc.) refaz
toda a cadeia de transformações do início — relendo o CSV do disco, removendo nulos,
convertendo datas, etc.

Como `df_limpo` é a base de todas as 8 perguntas, cachear aqui evita esse reprocessamento
repetido a cada consulta. O ganho é especialmente visível a partir da segunda ação.

```
Sem cache:  P1 → lê CSV → limpa → agrupa
            P2 → lê CSV → limpa → filtra
            P3 → lê CSV → limpa → calcula stddev  (3x o mesmo trabalho)

Com cache:  P1 → usa memória → agrupa
            P2 → usa memória → filtra
            P3 → usa memória → calcula stddev      (leitura só na primeira vez)
```

O `.persist()` é similar mas permite escolher onde armazenar — memória, disco ou ambos.
O `.cache()` é um atalho para o nível padrão que é memória.


In [5]:
# Medindo tempo SEM cache (primeira ação força o processamento)
_t_sem_cache = time.time()
_ = df_limpo.count()
tempo_sem_cache = time.time() - _t_sem_cache
print(f"[Comparativo] Tempo SEM cache (1ª contagem): {tempo_sem_cache:.2f}s")

# Aplicando cache — a partir daqui o DataFrame fica em memória
df_limpo.cache()
_ = df_limpo.count()
# Medindo tempo COM cache (dados já estão em memória)
_t_com_cache = time.time()
_ = df_limpo.count()
tempo_com_cache = time.time() - _t_com_cache
print(f"[Comparativo] Tempo COM cache (2ª contagem): {tempo_com_cache:.2f}s")
print(f"[Comparativo] Ganho com cache: {tempo_sem_cache / max(tempo_com_cache, 0.01):.1f}x mais rápido")

[Comparativo] Tempo SEM cache (1ª contagem): 4.95s


[Comparativo] Tempo COM cache (2ª contagem): 0.41s
[Comparativo] Ganho com cache: 12.0x mais rápido


### 3.4 Leitura e limpeza da base de CO₂

Utilizamos o dataset `owid-co2-data.csv` (Our World in Data), que cobre desde 1750
e contém colunas como `co2_per_capita` e `total_ghg`, conforme especificado no enunciado.

Filtramos:
- Registros anteriores a 1900
- Entradas que representam agregados regionais/globais (ex: "World", "Asia (GCP)")
- Linhas com `co2` nulo


In [6]:
_t = time.time()

df_co2 = spark.read.csv(
    "/opt/spark-data/input/owid-co2-data.csv",
    header=True,
    inferSchema=True,
)

agregados_globais = [
    "World", "Asia", "Asia (GCP)", "Asia (excl. China and India)",
    "Europe", "Europe (GCP)", "Europe (excl. EU-27)", "Europe (excl. EU-28)",
    "European Union (27)", "European Union (28)",
    "Africa", "Africa (GCP)",
    "North America", "North America (GCP)", "North America (excl. USA)",
    "South America", "South America (GCP)", "Central America (GCP)",
    "Oceania", "Oceania (GCP)",
    "High-income countries", "Low-income countries",
    "Lower-middle-income countries", "Upper-middle-income countries",
    "OECD (GCP)", "OECD (Jones et al.)", "Non-OECD (GCP)",
]

df_co2_clean = (
    df_co2.filter(col("year") >= 1900)
    .filter(~col("country").isin(agregados_globais))
    .select(
        col("country").alias("Country_CO2"),
        col("year").alias("Year_CO2"),
        col("co2"),
        col("co2_per_capita"),
        col("total_ghg"),
    )
    .dropna(subset=["co2"])
)

print(f"Registros CO₂ após limpeza: {df_co2_clean.count():,}")
df_co2_clean.show(5)
print(f"[Tempo] Leitura e limpeza CO₂: {time.time() - _t:.2f}s")

Registros CO₂ após limpeza: 20,164
+-----------+--------+-----+--------------+---------+
|Country_CO2|Year_CO2|  co2|co2_per_capita|total_ghg|
+-----------+--------+-----+--------------+---------+
|Afghanistan|    1949|0.015|         0.002|   18.667|
|Afghanistan|    1950|0.084|         0.011|   19.869|
|Afghanistan|    1951|0.092|         0.012|   21.069|
|Afghanistan|    1952|0.092|         0.011|   22.094|
|Afghanistan|    1953|0.106|         0.013|   23.256|
+-----------+--------+-----+--------------+---------+
only showing top 5 rows

[Tempo] Leitura e limpeza CO₂: 3.52s


## 4. P5 — Qualidade de Dados: Filtro de Incerteza

Identificamos registros onde `AverageTemperatureUncertainty` é superior a **10% da média
histórica** de temperatura da cidade.

Usamos uma **Window Function** particionada por `City` e `Country` para calcular a média
histórica de cada cidade sem precisar de um join separado.


In [7]:
_t = time.time()

# Especificação da janela por cidade
window_city = Window.partitionBy("City", "Country")

# Adicionando coluna com a média histórica de cada cidade
df_limpo_com_histavg = df_limpo.withColumn(
    "HistAvg", avg("AverageTemperature").over(window_city)
)

# Registros com alta incerteza (serão removidos)
df_alta_incerteza = df_limpo_com_histavg.filter(
    col("AverageTemperatureUncertainty") > (0.10 * col("HistAvg"))
)

print("P5 — Top 10 países com mais registros de alta incerteza:")
df_alta_incerteza.groupBy("Country").count().orderBy(col("count").desc()).show(10)

# DataFrame filtrado — usado em todas as perguntas seguintes
df_filtrado = df_limpo_com_histavg.filter(
    col("AverageTemperatureUncertainty") <= (0.10 * col("HistAvg"))
)

total = df_filtrado.count()
print(f"Registros restantes após filtro de incerteza: {total:,}")
print(f"[Tempo] P5 Qualidade de dados: {time.time() - _t:.2f}s")

P5 — Top 10 países com mais registros de alta incerteza:


+--------------+------+
|       Country| count|
+--------------+------+
|        Russia|344520|
|         China|256988|
| United States|191539|
|       Germany|115246|
|United Kingdom| 93975|
|        Poland| 60132|
|         Japan| 59784|
|         Spain| 57294|
|        Turkey| 55732|
|         Italy| 51719|
+--------------+------+
only showing top 10 rows



[Stage 34:=========================================>                (5 + 2) / 7]

Registros restantes após filtro de incerteza: 6,296,146
[Tempo] P5 Qualidade de dados: 11.65s


## 5. Análises — Respostas às Perguntas

### P1 — Média Móvel de Temperatura por Década

Calculamos a temperatura média global agrupando os registros por década.
A coluna `Decada` é criada dividindo o ano por 10, truncando para inteiro e multiplicando
por 10 novamente (ex: 1987 → 1980).


In [8]:
_t = time.time()

df_decada = (
    df_filtrado
    .withColumn("Decada", (col("Year") / 10).cast("int") * 10)
    .groupBy("Decada")
    .agg(_round(avg("AverageTemperature"), 2).alias("AvgGlobalTemp"))
    .orderBy("Decada")
)

print("P1 — Temperatura média global por década:")
df_decada.show(50)
print(f"[Tempo] P1: {time.time() - _t:.2f}s")

P1 — Temperatura média global por década:


[Stage 40:=================================>                        (4 + 3) / 7]

+------+-------------+
|Decada|AvgGlobalTemp|
+------+-------------+
|  1740|        22.47|
|  1750|        20.91|
|  1760|        20.19|
|  1770|        23.75|
|  1780|        23.21|
|  1790|         24.9|
|  1800|        25.63|
|  1810|        24.47|
|  1820|        24.65|
|  1830|        23.55|
|  1840|         22.2|
|  1850|        21.94|
|  1860|        20.52|
|  1870|        20.59|
|  1880|        19.28|
|  1890|        19.13|
|  1900|         18.8|
|  1910|        18.68|
|  1920|        18.79|
|  1930|        18.79|
|  1940|        18.81|
|  1950|        18.73|
|  1960|        18.61|
|  1970|        18.55|
|  1980|        18.67|
|  1990|        18.93|
|  2000|        19.21|
|  2010|        19.45|
+------+-------------+

[Tempo] P1: 4.43s


#### Gráfico P1 — Curva de Aquecimento Global por Década

In [9]:
# Coletando dados para o gráfico
dados_p1 = df_decada.toPandas()
dados_p1 = dados_p1[dados_p1["Decada"] >= 1850]  # foco no período moderno

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dados_p1["Decada"], dados_p1["AvgGlobalTemp"], marker="o", color="#2196F3", linewidth=2, markersize=5)
ax.fill_between(dados_p1["Decada"], dados_p1["AvgGlobalTemp"], alpha=0.15, color="#2196F3")
ax.set_title("Evolução da Temperatura Média Global por Década (1850–2010)", fontsize=14, fontweight="bold")
ax.set_xlabel("Década")
ax.set_ylabel("Temperatura Média (°C)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p1_temperatura_decadas.png", dpi=150)
plt.show()
print("Gráfico P1 salvo.")

Gráfico P1 salvo.


### P2 — Anomalias Locais: 10 Anos Mais Quentes por Continente

Para cada continente, filtramos países representativos e identificamos os 10 anos com
maior temperatura média nos últimos 50 anos (a partir de 1974).

> O dataset não possui coluna de continente, por isso usamos países representativos
> como proxy para cada região.


In [10]:
_t = time.time()

df_recente = df_filtrado.filter(col("Year") >= 1974)

continentes = {
    "America do Sul":      ["Brazil", "Argentina", "Chile", "Colombia", "Peru", "Venezuela",
                            "Ecuador", "Bolivia", "Paraguay", "Uruguay", "Guyana", "Suriname"],
    "America do Norte":    ["United States", "Canada", "Mexico"],
    "America Central e Caribe": ["Guatemala", "Honduras", "El Salvador", "Nicaragua", "Costa Rica",
                            "Panama", "Cuba", "Haiti", "Dominican Republic", "Jamaica",
                            "Puerto Rico", "Bahamas"],
    "Europa":              ["Germany", "France", "United Kingdom", "Spain", "Italy", "Poland",
                            "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", "Finland",
                            "Portugal", "Greece", "Czech Republic", "Romania", "Hungary", "Austria",
                            "Switzerland", "Bulgaria", "Serbia", "Croatia", "Slovakia", "Slovenia",
                            "Lithuania", "Latvia", "Estonia", "Iceland", "Ireland", "Albania",
                            "Bosnia And Herzegovina", "Macedonia", "Montenegro", "Moldova",
                            "Belarus", "Ukraine", "Cyprus"],
    "Asia":                ["China", "India", "Japan", "Russia", "Indonesia", "Pakistan", "Bangladesh",
                            "Vietnam", "Thailand", "Malaysia", "Philippines", "South Korea", "Taiwan",
                            "Hong Kong", "Singapore", "Sri Lanka", "Nepal", "Cambodia", "Laos", "Burma",
                            "Afghanistan", "Kazakhstan", "Uzbekistan", "Turkmenistan", "Tajikistan",
                            "Azerbaijan", "Georgia", "Armenia", "Iran", "Iraq", "Saudi Arabia", "Yemen",
                            "Oman", "Qatar", "United Arab Emirates", "Bahrain", "Jordan", "Lebanon",
                            "Israel", "Syria", "Turkey"],
    "Africa":              ["Nigeria", "South Africa", "Ethiopia", "Egypt", "Kenya", "Tanzania",
                            "Uganda", "Algeria", "Morocco", "Sudan", "Angola", "Mozambique", "Ghana",
                            "Cameroon", "Côte D'Ivoire", "Madagascar", "Niger", "Burkina Faso", "Mali",
                            "Malawi", "Zambia", "Zimbabwe", "Senegal", "Guinea", "Rwanda", "Benin",
                            "Burundi", "Somalia", "Togo", "Sierra Leone", "Libya", "Congo",
                            "Congo (Democratic Republic Of The)", "Gabon", "Eritrea", "Djibouti",
                            "Gambia", "Guinea Bissau", "Equatorial Guinea", "Central African Republic",
                            "Chad", "Liberia", "Mauritania", "Namibia", "Botswana", "Lesotho",
                            "Swaziland", "Reunion", "Mauritius"],
    "Oceania":             ["Australia", "New Zealand", "Papua New Guinea"],
}

dados_p2 = {}
for continente, paises in continentes.items():
    print(f"\nP2 — 10 anos mais quentes >> {continente}:")
    df_cont = (
        df_recente.filter(col("Country").isin(paises))
        .groupBy("Year")
        .agg(_round(avg("AverageTemperature"), 2).alias("AvgTemp"))
        .orderBy(col("AvgTemp").desc())
        .limit(10)
    )
    df_cont.show()
    dados_p2[continente] = df_cont.toPandas()

print(f"[Tempo] P2: {time.time() - _t:.2f}s")


P2 — 10 anos mais quentes >> America do Sul:


+----+-------+
|Year|AvgTemp|
+----+-------+
|1998|  22.15|
|2002|  22.15|
|2012|  22.11|
|1997|  22.07|
|2009|  22.01|
|2003|   22.0|
|2010|  21.98|
|2006|  21.97|
|2005|  21.97|
|2001|  21.96|
+----+-------+


P2 — 10 anos mais quentes >> America do Norte:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2013|  17.49|
|2012|   17.0|
|1998|  16.59|
|2006|  16.58|
|2011|  16.34|
|2005|  16.33|
|1990|  16.27|
|1999|  16.27|
|2002|  16.27|
|2007|  16.26|
+----+-------+


P2 — 10 anos mais quentes >> America Central e Caribe:
+----+-------+
|Year|AvgTemp|
+----+-------+
|1998|  25.89|
|1997|  25.85|
|2003|   25.7|
|2013|  25.65|
|2002|   25.6|
|1994|  25.58|
|2006|  25.58|
|2007|  25.58|
|2005|  25.55|
|2009|  25.53|
+----+-------+


P2 — 10 anos mais quentes >> Europa:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2007|  11.45|
|2011|  11.37|
|2013|  11.34|
|2000|  11.34|
|1990|  11.28|
|2002|  11.24|
|2006|  11.22|
|1989|  11.21|
|1994|   11.2|
|2008|  11.17|
+----+-------+


P2 — 10 anos 

+----+-------+
|Year|AvgTemp|
+----+-------+
|2013|   22.1|
|2010|  20.78|
|2012|  20.72|
|1998|   20.7|
|2002|  20.63|
|2007|  20.59|
|2009|  20.56|
|2004|  20.45|
|2006|  20.45|
|1999|   20.4|
+----+-------+




P2 — 10 anos mais quentes >> Africa:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2010|  24.35|
|2005|  24.16|
|1998|  24.09|
|2009|  24.05|
|2003|  24.03|
|2013|  23.99|
|1987|  23.96|
|2007|  23.95|
|2002|  23.95|
|2006|  23.92|
+----+-------+


P2 — 10 anos mais quentes >> Oceania:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2013|  16.89|
|1998|  16.75|
|2010|  16.63|
|2005|  16.62|
|2007|   16.6|
|1988|  16.59|
|1999|   16.5|
|2001|  16.46|
|2009|  16.45|
|2011|   16.4|
+----+-------+

[Tempo] P2: 16.88s


In [11]:
# Anos mais quentes
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
colors = cm.RdYlBu_r(np.linspace(0.3, 1.0, 10))

for i, (continente, df_pd) in enumerate(dados_p2.items()):
    df_pd_sorted = df_pd.sort_values("AvgTemp", ascending=True)
    ax = axes[i]
    ax.barh(df_pd_sorted["Year"].astype(str), df_pd_sorted["AvgTemp"], color=colors)
    ax.set_title(continente, fontsize=10, fontweight="bold")
    ax.set_xlabel("Temp. Média (°C)", fontsize=8)
    ax.grid(True, axis="x", alpha=0.3)

axes[-1].set_visible(False)  # oculta o subplot vazio (7 continentes, 8 slots)
fig.suptitle("Top 10 Anos Mais Quentes por Continente (1974–2013)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p2_anos_quentes_continente.png", dpi=150)
plt.show()
print("Gráfico P2 salvo.")

Gráfico P2 salvo.


### P3 — Cidades em Risco: Maior Desvio Padrão de Temperatura

O desvio padrão mede a **instabilidade climática** de uma cidade — quanto maior,
mais variável é a temperatura ao longo do tempo.

Filtramos registros a partir de 1920 (último século) e ordenamos pelo maior desvio padrão.


In [12]:
_t = time.time()

df_risk = (
    df_filtrado.filter(col("Year") >= 1920)
    .groupBy("City", "Country")
    .agg(_round(stddev("AverageTemperature"), 2).alias("Temperature_StdDev"))
    .orderBy(col("Temperature_StdDev").desc())
    .limit(10)
)

print("P3 — Top 10 cidades com maior instabilidade climática:")
df_risk.show()
print(f"[Tempo] P3: {time.time() - _t:.2f}s")

P3 — Top 10 cidades com maior instabilidade climática:


+-----------+-------+------------------+
|       City|Country|Temperature_StdDev|
+-----------+-------+------------------+
|     Yichun|  China|             14.34|
|   Tongliao|  China|             13.17|
|       Anda|  China|             12.94|
|      Hulan|  China|             12.94|
|       Fuyu|  China|             12.94|
|   Longfeng|  China|             12.94|
|     Harbin|  China|             12.94|
|    Qianguo|  China|             12.94|
|   Honggang|  China|             12.94|
|Shuangcheng|  China|             12.94|
+-----------+-------+------------------+

[Tempo] P3: 3.72s


#### Gráfico P3 — Cidades com Maior Instabilidade Climática

In [13]:
dados_p3 = df_risk.toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    dados_p3["City"] + " (" + dados_p3["Country"] + ")",
    dados_p3["Temperature_StdDev"],
    color="#E53935"
)
ax.set_title("Top 10 Cidades com Maior Desvio Padrão de Temperatura (desde 1920)", fontsize=13, fontweight="bold")
ax.set_xlabel("Desvio Padrão (°C)")
ax.invert_yaxis()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p3_cidades_risco.png", dpi=150)
plt.show()
print("Gráfico P3 salvo.")

Gráfico P3 salvo.


### P4 — Correlação de Estações em Zonas Tropicais

Calculamos a **correlação de Pearson** entre temperatura máxima e mínima mensal
em cidades localizadas entre as latitudes -23.5° e 23.5° (zona tropical).

> **Nota:** O dataset não fornece valores diários de mínima/máxima. `T_Max` e `T_Min`
> representam, respectivamente, o maior e menor valor mensal de `AverageTemperature`
> agrupados por cidade e ano.


In [14]:
_t = time.time()

df_tropicos = (
    df_filtrado
    .filter((col("Latitude") >= -23.5) & (col("Latitude") <= 23.5))
    .groupBy("City", "Year")
    .agg(
        _max("AverageTemperature").alias("T_Max"),
        _min("AverageTemperature").alias("T_Min"),
    )
)

corr_t = df_tropicos.stat.corr("T_Max", "T_Min")
print(f"P4 — Coeficiente de Correlação de Pearson (T_Max vs T_Min) nas zonas tropicais: {corr_t:.4f}")
print(f"[Tempo] P4: {time.time() - _t:.2f}s")

[Stage 149:========================>                                (3 + 4) / 7]

P4 — Coeficiente de Correlação de Pearson (T_Max vs T_Min) nas zonas tropicais: 0.4026
[Tempo] P4: 4.17s


#### Gráfico P4 — Dispersão T_Max vs T_Min em Zonas Tropicais

In [15]:
dados_p4 = df_tropicos.sample(fraction=0.02, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(dados_p4["T_Max"], dados_p4["T_Min"], alpha=0.3, s=10, color="#FF9800")
ax.set_title(f"Correlação T_Max vs T_Min em Zonas Tropicais\n(Pearson r = {corr_t:.4f})", fontsize=13, fontweight="bold")
ax.set_xlabel("T_Max (°C)")
ax.set_ylabel("T_Min (°C)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p4_correlacao_tropicais.png", dpi=150)
plt.show()
print("Gráfico P4 salvo.")

Gráfico P4 salvo.


### P6 — Correlação entre Emissões de CO₂ e Aquecimento (Join)

A pergunta do enunciado é sobre o **aumento** das emissões vs o **aumento** da temperatura
— ou seja, correlação de **tendências** ao longo do tempo, e não de níveis absolutos.
Abordamos em duas etapas:

- **Etapa 1 (diagnóstico):** `join` das bases por `Country + Year` e correlação de Pearson
  sobre os **níveis absolutos** de CO₂ e temperatura. Serve apenas como diagnóstico —
  sofre de viés geográfico (grandes emissores como Rússia/Canadá são países frios).
- **Etapa 2 (resposta):** correlação entre as **taxas de aumento** (o *slope* da regressão
  linear por país) de CO₂ e de temperatura, através dos países.

In [16]:
_t = time.time()

df_temp_country = (
    df_filtrado.filter(col("Year") >= 1974)
    .groupBy("Country", "Year")
    .agg(avg("AverageTemperature").alias("AvgTemp"))
)

df_join = df_temp_country.join(
    df_co2_clean,
    (df_temp_country["Country"] == df_co2_clean["Country_CO2"])
    & (df_temp_country["Year"] == df_co2_clean["Year_CO2"]),
    "inner",
).dropna(subset=["co2", "AvgTemp"])

print(f"Registros após join: {df_join.count():,}")
df_join.show(5)

corr_co2 = df_join.stat.corr("co2", "AvgTemp")
print(f"\nP6 — Coeficiente de Correlação de Pearson (CO₂ vs Temperatura Média): {corr_co2:.4f}")
print(f"[Tempo] P6: {time.time() - _t:.2f}s")

Registros após join: 5,829


+-----------+----+------------------+-----------+--------+--------+--------------+---------+
|    Country|Year|           AvgTemp|Country_CO2|Year_CO2|     co2|co2_per_capita|total_ghg|
+-----------+----+------------------+-----------+--------+--------+--------------+---------+
|     Turkey|2010|15.681119999999998|     Turkey|    2010|  318.71|         4.345|  397.616|
|     Brazil|1997|22.880382196969684|     Brazil|    1997| 306.949|         1.842| 2696.838|
| Azerbaijan|2012|13.890909090909092| Azerbaijan|    2012|   32.77|         3.495|    50.62|
|     Russia|1979| 6.665930343511452|     Russia|    1979|2049.709|        14.873| 2928.791|
|Switzerland|2001| 8.521899999999997|Switzerland|    2001|  45.085|         6.239|   54.617|
+-----------+----+------------------+-----------+--------+--------+--------------+---------+
only showing top 5 rows



[Stage 184:>                                                        (0 + 6) / 7]


P6 — Coeficiente de Correlação de Pearson (CO₂ vs Temperatura Média): -0.1727
[Tempo] P6: 9.80s


#### Gráfico P6 (Etapa 1) — Dispersão CO₂ vs Temperatura (níveis absolutos, diagnóstico)

In [17]:
dados_p6 = df_join.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    dados_p6["co2"],
    dados_p6["AvgTemp"],
    alpha=0.4, s=15,
    c=dados_p6["Year"], cmap="RdYlBu_r"
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Ano")
ax.set_title(f"CO₂ vs Temperatura Média por País (1974–2013)\n(Pearson r = {corr_co2:.4f})", fontsize=13, fontweight="bold")
ax.set_xlabel("Emissões de CO₂ (Mt)")
ax.set_ylabel("Temperatura Média (°C)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p6_co2_temperatura.png", dpi=150)
plt.show()
print("Gráfico P6 salvo.")

Gráfico P6 salvo.


#### Etapa 2 — Correlação das taxas de aumento (resposta à pergunta)

A pergunta pede o **aumento** — então calculamos, para **cada país**, a **tendência**
(inclinação da regressão linear) da temperatura e das emissões ao longo dos anos, usando a
forma fechada `slope = covar(ano, x) / var(ano)` com as funções `covar_pop` e `var_pop`.
Em seguida correlacionamos as duas tendências **entre os países**. Assim removemos o viés
de nível absoluto e medimos de fato se "quem aumentou emissão aqueceu mais".

O resultado (≈ 0) é cientificamente coerente: o CO₂ é um gás **globalmente bem misturado**,
então o aquecimento de um país é determinado pelo forçamento **global** (e efeitos
regionais como a amplificação ártica), não pelas suas emissões locais. Rússia e Ucrânia,
por exemplo, aqueceram entre os mais rápidos mesmo com as emissões **caindo** após o
colapso da União Soviética.

In [18]:
# Etapa 2 — Correlação das TAXAS DE AUMENTO (resposta à pergunta do enunciado)
# slope (inclinação da regressão linear) por país = covar(ano, x) / var(ano)
from pyspark.sql.functions import covar_pop, var_pop

_t = time.time()

slopes = (
    df_join.groupBy("Country").agg(
        (covar_pop("Year", "AvgTemp") / var_pop("Year")).alias("slope_temp"),
        (covar_pop("Year", "co2") / var_pop("Year")).alias("slope_co2"),
        count("*").alias("n"),
    )
    .filter(col("n") >= 10)  # exige no mínimo 10 anos de dados por país
)

corr_variacao = slopes.stat.corr("slope_temp", "slope_co2")
n_paises = slopes.count()

print(f"P6 (Etapa 2) — Pearson entre taxas de aumento (dCO2 vs dTemp): {corr_variacao:.4f}")
print(f"Países considerados: {n_paises}")
print("\nPaíses que mais aqueceram (maior tendência de temperatura):")
slopes.select(
    "Country",
    _round("slope_temp", 4).alias("dTemp_por_ano"),
    _round("slope_co2", 3).alias("dCO2_por_ano"),
    "n",
).orderBy(col("slope_temp").desc()).show(10, truncate=False)
print(f"[Tempo] P6 Etapa 2: {time.time() - _t:.2f}s")

P6 (Etapa 2) — Pearson entre taxas de aumento (dCO2 vs dTemp): -0.0017
Países considerados: 147

Países que mais aqueceram (maior tendência de temperatura):


[Stage 234:>                                                        (0 + 6) / 7]

+--------------------+-------------+------------+---+
|Country             |dTemp_por_ano|dCO2_por_ano|n  |
+--------------------+-------------+------------+---+
|Kazakhstan          |0.093        |-0.532      |40 |
|Finland             |0.0909       |0.382       |40 |
|Sweden              |0.0674       |-0.787      |40 |
|Russia              |0.0638       |-17.555     |40 |
|Azerbaijan          |0.0559       |-0.521      |40 |
|Ukraine             |0.0515       |-12.675     |40 |
|Eritrea             |0.0514       |-0.015      |20 |
|United Arab Emirates|0.0507       |4.116       |40 |
|Turkmenistan        |0.0483       |0.864       |40 |
|Qatar               |0.047        |1.948       |40 |
+--------------------+-------------+------------+---+
only showing top 10 rows

[Tempo] P6 Etapa 2: 10.37s


In [19]:
# Gráfico Etapa 2 — tendência de emissão vs tendência de temperatura por país
dados_slopes = slopes.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dados_slopes["slope_co2"], dados_slopes["slope_temp"],
           alpha=0.55, s=24, color="#2E7D32", edgecolors="white", linewidths=0.3)
ax.axhline(0, color="gray", lw=0.6)
ax.axvline(0, color="gray", lw=0.6)
ax.set_xlim(-25, 55)  # China (+205 Mt/ano) fica fora de escala para legibilidade

for _, row in dados_slopes.iterrows():
    if row["slope_co2"] > 55:
        ax.annotate(f'{row["Country"]} (+{row["slope_co2"]:.0f} Mt/ano)',
                    xy=(54, row["slope_temp"]), xytext=(40, row["slope_temp"] + 0.012),
                    fontsize=8, color="#B71C1C",
                    arrowprops=dict(arrowstyle="->", color="#B71C1C", lw=0.7))
    if row["slope_co2"] < -10:
        ax.annotate(row["Country"], xy=(row["slope_co2"], row["slope_temp"]),
                    fontsize=7, color="#1565C0")

ax.set_title(f"P6 (Etapa 2) — Aumento de CO₂ vs Aumento de Temperatura por país (1974–2013)\n"
             f"(Pearson r = {corr_variacao:.4f}, n = {n_paises} países)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Tendência de emissões de CO₂ (Mt/ano)")
ax.set_ylabel("Tendência de temperatura (°C/ano)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p6_variacao.png", dpi=150)
plt.show()
print("Gráfico P6 (Etapa 2) salvo.")

Gráfico P6 (Etapa 2) salvo.


### P7 — Ranking de Aceleração Térmica (Window Functions)

Calculamos a temperatura média por país e por década, depois usamos a função `lag()`
para comparar cada década com a anterior dentro do mesmo país.

A **aceleração** é a diferença entre a temperatura da década de 2010 e a de 2000.
Os 10 países com maior aceleração são listados.


In [20]:
_t = time.time()

df_decada_pais = (
    df_filtrado
    .withColumn("Decada", (col("Year") / 10).cast("int") * 10)
    .groupBy("Country", "Decada")
    .agg(avg("AverageTemperature").alias("TempDecada"))
)

window_accel = Window.partitionBy("Country").orderBy("Decada")

df_accel = (
    df_decada_pais
    .withColumn("TempAnterior", lag("TempDecada", 1).over(window_accel))
    .withColumn("Aceleracao", col("TempDecada") - col("TempAnterior"))
    .filter(col("Decada") == 2010)
    .orderBy(col("Aceleracao").desc())
    .limit(10)
)

print("P7 — Top 10 países com maior aceleração térmica (2000 → 2010):")
df_accel.select("Country", _round("Aceleracao", 3).alias("Graus_Aceleracao")).show()
print(f"[Tempo] P7: {time.time() - _t:.2f}s")

P7 — Top 10 países com maior aceleração térmica (2000 → 2010):


[Stage 255:================================>                        (4 + 3) / 7]

+----------+----------------+
|   Country|Graus_Aceleracao|
+----------+----------------+
|   Finland|           1.076|
|    Russia|           1.049|
|Kazakhstan|           0.778|
|    Canada|           0.767|
|      Iran|           0.761|
|      Iraq|           0.736|
|   Georgia|           0.727|
|   Armenia|           0.727|
|    Turkey|           0.663|
|Tajikistan|           0.639|
+----------+----------------+

[Tempo] P7: 3.21s


#### Gráfico P7 — Aceleração Térmica por País

In [21]:
dados_p7 = df_accel.select("Country", _round("Aceleracao", 3).alias("Graus_Aceleracao")).toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#D32F2F" if v > 0.8 else "#EF5350" if v > 0.6 else "#FF8A80" for v in dados_p7["Graus_Aceleracao"]]
ax.barh(dados_p7["Country"], dados_p7["Graus_Aceleracao"], color=colors)
ax.set_title("Top 10 Países com Maior Aceleração Térmica (Década 2000 → 2010)", fontsize=13, fontweight="bold")
ax.set_xlabel("Aceleração (°C por década)")
ax.invert_yaxis()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p7_aceleracao_termica.png", dpi=150)
plt.show()
print("Gráfico P7 salvo.")

Gráfico P7 salvo.


### P8 — Previsão de Tendência com Regressão Linear (MLlib)

Utilizamos **Regressão Linear** do Spark MLlib para prever a temperatura média de
São Paulo nos 5 anos seguintes ao último registro disponível no dataset (~2013).

O modelo é treinado com os dados de **1993 a 2013** (últimos 20 anos disponíveis).
O `VectorAssembler` converte a coluna `Year` em um vetor de features, formato
exigido pelo MLlib.


In [22]:
_t = time.time()

df_sp = (
    df_filtrado
    .filter((col("City") == "São Paulo") & (col("Year") >= 1993))
    .groupBy("Year")
    .agg(avg("AverageTemperature").alias("label"))
    .orderBy("Year")
)

print("Dados de treino (São Paulo):")
df_sp.show()

assembler = VectorAssembler(inputCols=["Year"], outputCol="features")
df_ml = assembler.transform(df_sp).select("features", "label")

lr = LinearRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(df_ml)

print(f"Coeficiente (inclinação): {lr_model.coefficients[0]:.6f}")
print(f"Intercepto: {lr_model.intercept:.4f}")

anos_futuros = spark.createDataFrame([(y,) for y in range(2014, 2019)], ["Year"])
anos_futuros_ml = assembler.transform(anos_futuros)
previsoes = lr_model.transform(anos_futuros_ml)

print("\nP8 — Temperatura prevista para São Paulo (2014–2018):")
previsoes.select("Year", _round("prediction", 2).alias("Temperatura_Prevista")).show()
print(f"[Tempo] P8: {time.time() - _t:.2f}s")

Dados de treino (São Paulo):
+----+------------------+
|Year|             label|
+----+------------------+
|1993|20.488416666666666|
|1994|20.770999999999997|
|1995|20.608583333333332|
|1996|20.176083333333334|
|1997|          20.77125|
|1998|20.714750000000002|
|1999|20.044416666666667|
|2000|20.316666666666666|
|2001|            20.964|
|2002|21.304000000000002|
|2003|20.755666666666666|
|2004|          20.04825|
|2005|20.783166666666666|
|2006|20.603916666666667|
|2007|20.933416666666666|
|2008|20.220666666666663|
|2009|20.775583333333334|
|2010|20.716583333333332|
|2011|20.330083333333338|
|2012| 21.01891666666667|
+----+------------------+
only showing top 20 rows



26/05/27 00:16:49 WARN Instrumentation: [4dfa43ad] regParam is zero, which might cause numerical instability and overfitting.
26/05/27 00:16:50 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/27 00:16:50 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coeficiente (inclinação): -0.000099
Intercepto: 20.7900

P8 — Temperatura prevista para São Paulo (2014–2018):


[Stage 315:==========================================>              (3 + 1) / 4]

+----+--------------------+
|Year|Temperatura_Prevista|
+----+--------------------+
|2014|               20.59|
|2015|               20.59|
|2016|               20.59|
|2017|               20.59|
|2018|               20.59|
+----+--------------------+

[Tempo] P8: 7.53s


#### Gráfico P8 — Histórico + Previsão de Temperatura em São Paulo

In [23]:
historico = df_sp.toPandas()
previsao = previsoes.select("Year", _round("prediction", 2).alias("Temperatura_Prevista")).toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(historico["Year"], historico["label"], marker="o", color="#1565C0", linewidth=2, label="Histórico (1993–2013)")
ax.plot(previsao["Year"], previsao["Temperatura_Prevista"], marker="s", color="#E53935",
        linewidth=2, linestyle="--", label="Previsão (2014–2018)")
ax.axvline(x=2013.5, color="gray", linestyle=":", alpha=0.7)
ax.set_title("Temperatura Média Anual em São Paulo — Histórico e Previsão MLlib", fontsize=13, fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Temperatura Média (°C)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/opt/spark-data/output/grafico_p8_previsao_saopaulo.png", dpi=150)
plt.show()
print("Gráfico P8 salvo.")

Gráfico P8 salvo.


## 6. Tempo Total de Execução

In [24]:
tempo_total = time.time() - _tempo_inicio_geral
minutos = int(tempo_total // 60)
segundos = tempo_total % 60
print(f"Pipeline completo executado em {minutos}min {segundos:.2f}s")

Pipeline completo executado em 2min 45.91s


## 7. Encerramento da Sessão

Encerramos a `SparkSession` para liberar os recursos do cluster.


In [25]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.
